# Galaxy Defender — Android APK (with icon)

This rebuild uses your **icon.png** as the app icon. It does **not** change any files in `kivy_app/`.

On your Redmi Pad 2:

1. Sign in with Google if asked.
2. Tap **Runtime → Run all**.
3. Wait **25–50 minutes**. Keep this tab open and the pad plugged in.
4. When the last cell finishes, it downloads `galaxydefender-1.1-arm64-v8a-debug.apk`.
5. Open that file → **Install**. If Android says the app exists, uninstall the old Galaxy Defender first, then install this one.


In [ ]:
from IPython.display import Javascript, display
display(Javascript('''
function KeepAlive(){
  console.log("colab keep-alive");
  const b = document.querySelector("colab-connect-button") || document.querySelector("#connect");
  if (b) b.click();
}
setInterval(KeepAlive, 60000);
'''))

import os, sys, subprocess
print("Notebook Python", sys.version)

def sh(cmd):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.check_call(cmd, shell=isinstance(cmd, str))

sh("sudo apt-get update -qq")
sh("sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
   "python3.11 python3.11-venv python3.11-dev "
   "build-essential git zip unzip autoconf libtool pkg-config "
   "openjdk-17-jdk zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 "
   "cmake libffi-dev libssl-dev wget || true")
sh("sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq python3.11 python3.11-venv python3.11-dev openjdk-17-jdk")

java_home = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + "/bin:" + os.environ.get("PATH", "")
sh(["java", "-version"])

venv = "/content/bz"
if not os.path.isfile(venv + "/bin/python"):
    sh(["python3.11", "-m", "venv", venv])
pip = venv + "/bin/pip"
sh([pip, "install", "-q", "-U", "pip", "wheel"])
sh([pip, "install", "-q", "Cython==0.29.36", "setuptools<71.0.0", "buildozer", "virtualenv", "sh", "pexpect", "Pillow"])
os.environ["PATH"] = venv + "/bin:" + os.environ["PATH"]
sh(["buildozer", "--version"])
print("Setup done")

In [ ]:
import os, glob, shutil, subprocess
from pathlib import Path

ROOT = "/content/Galaxy-Defender"
if os.path.isdir(ROOT):
    shutil.rmtree(ROOT)

subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/AmarGams/Galaxy-Defender.git", ROOT])
os.chdir(ROOT)
print("cwd", os.getcwd())

# Never edit Kivy game files.
kivy_main = Path("kivy_app/main.py")
kivy_icon = Path("kivy_app/icon.png")
root_icon = Path("icon.png")
assert kivy_main.is_file(), "kivy_app/main.py missing — clone failed"
assert kivy_icon.is_file() or root_icon.is_file(), "icon.png missing"
print("kivy files left untouched")

# Make a 512x512 launcher from your icon without changing kivy_app/.
src_icon = kivy_icon if kivy_icon.is_file() else root_icon
try:
    from PIL import Image
    im = Image.open(src_icon).convert("RGB").resize((512, 512), Image.Resampling.LANCZOS)
    im.save("icon_launcher.png", format="PNG", optimize=True, compress_level=9)
    print("Wrote icon_launcher.png from", src_icon, "bytes", Path("icon_launcher.png").stat().st_size)
except Exception as e:
    print("Pillow resize skipped:", e)
    if not Path("icon_launcher.png").is_file():
        shutil.copyfile(src_icon, "icon_launcher.png")

spec_path = Path("buildozer.spec")
assert spec_path.is_file(), "buildozer.spec missing"
spec = spec_path.read_text(encoding="utf-8")
if "icon.filename" not in spec:
    spec = spec.replace("[app]", "[app]\nicon.filename = icon_launcher.png\npresplash.filename = icon_launcher.png", 1)
    spec_path.write_text(spec, encoding="utf-8")
    print("Added icon.filename to spec")
else:
    print("Spec already has an icon line — keeping it")
print(spec_path.read_text(encoding="utf-8"))

env = os.environ.copy()
env["PATH"] = "/content/bz/bin:/usr/lib/jvm/java-17-openjdk-amd64/bin:" + env.get("PATH", "")
env["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
print("Starting buildozer (25–50 min first time)…")
r = subprocess.run(["buildozer", "-v", "android", "debug"], env=env)
apks = glob.glob("bin/*.apk")
print("exit", r.returncode, "APKs", apks)
if r.returncode != 0 or not apks:
    raise SystemExit("Build failed. Scroll up for # command failed / Error compiling.")

In [ ]:
import glob, os
from google.colab import files

apks = sorted(glob.glob("/content/Galaxy-Defender/bin/*.apk"))
print("Ready:", apks)
if not apks:
    raise SystemExit("No APK to download. The build cell did not succeed.")
for apk in apks:
    print(apk, round(os.path.getsize(apk) / 1024 / 1024, 1), "MB")
    files.download(apk)